# 05 — NLP Sentimen (Transformers)

Tujuan:
- Menjalankan inference sentimen dengan pipeline Transformers
- (Opsional) Fine-tuning ringan pada subset IMDB
- Evaluasi metrik sederhana (Accuracy/F1)

Catatan:
- Unduhan model/dataset memerlukan internet. Untuk offline gunakan model lokal/di-cache.
- Untuk eksperimen cepat, gunakan subset kecil agar runtime singkat.


In [ ]:
# Cek versi dan import dasar
import transformers, datasets, numpy as np
from transformers import pipeline
transformers.__version__, datasets.__version__

## 1) Inference cepat (tanpa training)
Menggunakan pipeline built-in untuk analisis sentimen.

In [ ]:
clf = pipeline("sentiment-analysis")
texts = [
    "Produknya memuaskan, pengiriman cepat.",
    "Sangat mengecewakan, tidak sesuai deskripsi.",
    "Biasa saja, tapi fungsional.",
]
for t in texts:
    print(t, "->", clf(t))

## 2) Fine-tuning ringan pada IMDB (subset kecil)
Perhatian: Ini hanya contoh quick-start, gunakan subset kecil agar cepat.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

ds = load_dataset("imdb")
train_small = ds["train"].shuffle(seed=42).select(range(1000))
test_small  = ds["test"].shuffle(seed=42).select(range(1000))

model_name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=128)

train_tok = train_small.map(tokenize, batched=True)
test_tok  = test_small.map(tokenize, batched=True)
train_tok = train_tok.remove_columns(["text"])
test_tok  = test_tok.remove_columns(["text"])
train_tok = train_tok.rename_column("label", "labels")
test_tok  = test_tok.rename_column("label", "labels")
train_tok.set_format("torch")
test_tok.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    f1 = f1_score(p.label_ids, preds)
    return {"accuracy": acc, "f1": f1}

args = TrainingArguments(
    output_dir="./tmp-imdb",
    evaluation_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tok,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

## 3) Gunakan model terlatih untuk inference singkat (opsional)
Trainer akan menyimpan checkpoint di `./tmp-imdb` (jika diaktifkan). Untuk demo ini kita tidak menyimpan.
Anda bisa langsung memanggil `trainer.predict()` pada sampel kecil untuk melihat output logits/prediksi.

In [ ]:
pred_out = trainer.predict(test_tok.select(range(10)))
preds = pred_out.predictions.argmax(axis=1)
list(zip(preds.tolist(), pred_out.label_ids.tolist()))

Selesai. Untuk penggunaan di aplikasi (mis. FastAPI), Anda bisa memanggil pipeline atau `AutoModelForSequenceClassification` langsung. 
Untuk produksi, pertimbangkan model yang lebih kecil/ter-quantize agar cepat dan hemat biaya.